# TicketStream - Week 3  API Notebook
**CoreSmart GenAI Developer Course · Week 3 · **

TicketStream takes a free-text support message, extracts a Pydantic-validated ticket
(discriminated intent union, Literal priority), and streams the validated fields
back as Server-Sent Events. A routing tool fires after the schema gate passes.

SSE frames use **Pattern B** (structured typed events), not token deltas:
`event: intent | priority | customer_id | routed | done | validation_failed`

Each endpoint is shown two ways:
- **cURL (Windows cmd)** - `%%cmd` cell magic, Windows double-quote syntax
- **Python** - `requests` library with SSE iteration

---
### Before you start
1. Server running: `uvicorn app.main:app --reload`
2. `.env` file with `OPENAI_API_KEY=sk-...`
3. Run the **Setup** cell below once.

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes.
> Streaming curl cells use the `-N` flag to disable buffering.

In [2]:
# Setup -- run this cell first
import requests, json, textwrap

BASE = 'http://localhost:8000'

# ASCII only -- no em dashes or Unicode (breaks Windows cmd curl)
DEMO_NOTES = 'Hi, order ORD-1042 was supposed to arrive Monday. Still no update. Customer ID 8821.'

# Extra demos covering all three intent types
DEMO = {
    'order':   'Hi, order ORD-1042 was supposed to arrive Monday. Still no update. Customer ID 8821.',
    'refund':  'I need a refund on order ORD-2222 -- it arrived damaged. Customer 9000.',
    'billing': 'I have a question about invoice INV-5050 -- I was charged twice.',
}

print('Setup complete.')
print('BASE:', BASE)

Setup complete.
BASE: http://localhost:8000


---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

> This section is identical across all weeks. Do not modify it.

In [3]:
%%cmd
curl -s http://localhost:8000/health

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 3/>curl -s http://localhost:8000/health
{"status":"ok","model":"gpt-5.4-mini-2026-03-17","models":{"openai":"gpt-5.4-mini-2026-03-17","nano":"gpt-5.4-nano-2026-03-17"}}
week 3/>

In [4]:
# Health check -- Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

Status : 200
{
  "status": "ok",
  "model": "gpt-5.4-mini-2026-03-17",
  "models": {
    "openai": "gpt-5.4-mini-2026-03-17",
    "nano": "gpt-5.4-nano-2026-03-17"
  }
}


---
## 2 · Submit Ticket - `POST /intake-stream` (SSE Pattern B)

Sends a support message. The server validates it into a `TicketSchema` and streams
the fields back as typed SSE events. The routing tool fires *after* the schema gate.

Key concept: **validate-first, stream-second** - partial JSON is not validatable.
The model produces the complete object, Pydantic validates once, then we stream the fields.

Request body:
```json
{ "message": "string", "provider": "openai" | "nano" }
```

SSE frame sequence:
```
event: intent       data: {"type": "order", "order_id": "ORD-1042"}
event: priority     data: "high"
event: customer_id  data: {"value": 8821}
event: routed       data: {"team": "fulfillment", "priority": "high", "ticket_url": "..."}
event: done         data: {"ok": true, "correction_count": 0, "attempts": 1, "model": "..."}
```

> Requires `OPENAI_API_KEY` in `.env`. Streaming curl uses `-N` to disable buffering.

In [5]:
%%cmd
curl -s -N -X POST http://localhost:8000/intake-stream -H "Content-Type: application/json" -d "{\"message\": \"Hi, order ORD-1042 was supposed to arrive Monday. Customer ID 8821.\", \"provider\": \"openai\"}"

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 3/>curl -s -N -X POST http://localhost:8000/intake-stream -H "Content-Type: application/json" -d "{\"message\": \"Hi, order ORD-1042 was supposed to arrive Monday. Customer ID 8821.\", \"provider\": \"openai\"}"
event: intent
data: {"type": "order", "order_id": "ORD-1042"}

event: priority
data: high

event: customer_id
data: {"value": 8821}

event: routed
data: {"team": "fulfillment", "priority": "high", "ticket_url": "https://tickets.internal/fulfillment/1780683735"}

event: done
data: {"ok": true, "model": "gpt-5.4-mini-2026-03-17", "correction_count": 0, "attempts": 1}


week 3/>

In [6]:
# POST /intake-stream -- Python (SSE Pattern B: structured events)
# Note: frames use named events (intent/priority/etc.), not {delta: "..."}
print('-- SSE event stream --\n')
with requests.post(f'{BASE}/intake-stream',
                   json={'message': DEMO_NOTES, 'provider': 'openai'},
                   stream=True) as resp:
    resp.raise_for_status()
    event_name = ''
    for raw_line in resp.iter_lines():
        if not raw_line:
            event_name = ''
            continue
        line = raw_line.decode() if isinstance(raw_line, bytes) else raw_line
        if line.startswith('event: '):
            event_name = line[7:].strip()
            continue
        if not line.startswith('data: '):
            continue
        payload_str = line[6:]
        try:
            data = json.loads(payload_str)
            print(f'  [{event_name:18}] {json.dumps(data)}')
            if event_name == 'done':
                print('\n-- Stream complete --')
                break
        except json.JSONDecodeError:
            pass

-- SSE event stream --

  [intent            ] {"type": "order", "order_id": "ORD-1042"}
  [customer_id       ] {"value": 8821}
  [routed            ] {"team": "fulfillment", "priority": "high", "ticket_url": "https://tickets.internal/fulfillment/1780683738"}
  [done              ] {"ok": true, "model": "gpt-5.4-mini-2026-03-17", "correction_count": 0, "attempts": 1}

-- Stream complete --


---
## 3 · Multi-Model Comparison - openai vs nano

Both providers run the same extraction pipeline. Compare correction counts and
the quality of the discriminated union classification.

In [7]:
# Multi-model comparison -- collect the done frame from each provider
def run_intake(message, provider):
    result = {'provider': provider, 'intent': None, 'priority': None, 'corrections': '?', 'model': '?'}
    with requests.post(f'{BASE}/intake-stream',
                       json={'message': message, 'provider': provider},
                       stream=True) as resp:
        if resp.status_code != 200:
            result['error'] = resp.json().get('detail', '?')
            return result
        event_name = ''
        for raw_line in resp.iter_lines():
            if not raw_line: event_name = ''; continue
            line = raw_line.decode() if isinstance(raw_line, bytes) else raw_line
            if line.startswith('event: '): event_name = line[7:].strip(); continue
            if not line.startswith('data: '): continue
            try:
                data = json.loads(line[6:])
                if event_name == 'intent':   result['intent'] = data.get('type')
                if event_name == 'priority': result['priority'] = data
                if event_name == 'done':
                    result['corrections'] = data.get('correction_count', 0)
                    result['model'] = data.get('model', '?')
                    break
            except json.JSONDecodeError: pass
    return result

print(f'Input: "{DEMO_NOTES}"\n')
print(f'{"provider":<10} {"intent":<10} {"priority":<10} {"corrections":<13} model')
print('-' * 68)
for provider in ['openai', 'nano']:
    r = run_intake(DEMO_NOTES, provider)
    if 'error' in r:
        print(f'{r["provider"]:<10} ERROR: {r["error"]}')
    else:
        print(f'{r["provider"]:<10} {str(r["intent"]):<10} {str(r["priority"]):<10} {str(r["corrections"]):<13} {r["model"]}')

Input: "Hi, order ORD-1042 was supposed to arrive Monday. Still no update. Customer ID 8821."

provider   intent     priority   corrections   model
--------------------------------------------------------------------
openai     order      None       0             gpt-5.4-mini-2026-03-17
nano       order      None       0             gpt-5.4-nano-2026-03-17


---
## 4 · Schema Validation - Discriminated Union

Send all three intent types. The discriminated union routes each message to the
correct `OrderIntent`, `RefundIntent`, or `BillingIntent` subclass.

In [8]:
# Test all three intent types
for label, message in DEMO.items():
    r_intent = None
    with requests.post(f'{BASE}/intake-stream',
                       json={'message': message, 'provider': 'openai'},
                       stream=True) as resp:
        if resp.status_code != 200: print(f'{label}: ERROR'); continue
        event_name = ''
        for raw_line in resp.iter_lines():
            if not raw_line: event_name = ''; continue
            line = raw_line.decode() if isinstance(raw_line, bytes) else raw_line
            if line.startswith('event: '): event_name = line[7:].strip(); continue
            if not line.startswith('data: '): continue
            try:
                data = json.loads(line[6:])
                if event_name == 'intent': r_intent = data; break
            except json.JSONDecodeError: pass
    print(f'{label:<10} -> intent: {json.dumps(r_intent)}')

order      -> intent: {"type": "order", "order_id": "ORD-1042"}
refund     -> intent: {"type": "refund", "order_id": "ORD-2222", "reason": "arrived damaged"}
billing    -> intent: {"type": "billing", "invoice_id": "INV-5050"}


---
## 5 · Failure Mode - Invalid Input (422)

Pydantic validates `message` with `min_length=1` before any API call is made.
An empty string returns **422** - no tokens spent.

> This failure pattern is identical across all weeks. Update the endpoint and payload only.

In [9]:
%%cmd
curl -s -X POST http://localhost:8000/intake-stream -H "Content-Type: application/json" -d "{\"message\": \"\"}"

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 3/>curl -s -X POST http://localhost:8000/intake-stream -H "Content-Type: application/json" -d "{\"message\": \"\"}"
{"detail":[{"type":"string_too_short","loc":["body","message"],"msg":"String should have at least 1 character","input":"","ctx":{"min_length":1}}]}
week 3/>

In [10]:
# Failure: empty message -- Python
r = requests.post(f'{BASE}/intake-stream', json={'message': ''})
print(f'Status: {r.status_code}  (expected 422 -- Pydantic rejects empty message, no API call made)')
print(json.dumps(r.json(), indent=2))

Status: 422  (expected 422 -- Pydantic rejects empty message, no API call made)
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "message"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}


---
## 6 · Failure Mode - Schema Reject with Retry-Correct (validation_failed SSE frame)

When the model produces an invalid ticket (e.g. priority outside the Literal enum),
the retry-with-correction loop fires. If all retries fail, the stream emits
`event: validation_failed` instead of the ticket fields.

The message below is intentionally ambiguous to push the model toward a
non-enum priority value like 'urgent'.

In [11]:
# Ambiguous intake that may push the model toward an invalid priority
# If the correction loop succeeds, you will see event:priority with a valid value.
# If it exhausts all retries, you will see event:validation_failed.
test_msg = 'SUPER URGENT DROP EVERYTHING my order ORD-9999 is missing!!!'
print(f'Sending: "{test_msg}"\n')

event_name = ''
with requests.post(f'{BASE}/intake-stream',
                   json={'message': test_msg, 'provider': 'openai'},
                   stream=True) as resp:
    resp.raise_for_status()
    for raw_line in resp.iter_lines():
        if not raw_line: event_name = ''; continue
        line = raw_line.decode() if isinstance(raw_line, bytes) else raw_line
        if line.startswith('event: '): event_name = line[7:].strip(); continue
        if not line.startswith('data: '): continue
        try:
            data = json.loads(line[6:])
            print(f'  [{event_name:18}] {json.dumps(data)}')
            if event_name in ('done', 'validation_failed'): break
        except json.JSONDecodeError: pass

Sending: "SUPER URGENT DROP EVERYTHING my order ORD-9999 is missing!!!"

  [intent            ] {"type": "order", "order_id": "ORD-9999"}
  [customer_id       ] {"value": null}
  [routed            ] {"team": "fulfillment", "priority": "critical", "ticket_url": "https://tickets.internal/fulfillment/1780683780"}
  [done              ] {"ok": true, "model": "gpt-5.4-mini-2026-03-17", "correction_count": 0, "attempts": 1}


---
## 7 · Failure Mode - Schema Correction, Deterministically (`tool_arg_validation_failed`, `schema_correction_exhausted`)

Section 6 pushes a live model toward an invalid priority - which is a
*statistical* event, so it may or may not fire on any given run. The next three
sections pin the same failure paths down deterministically: we drive the real
`app/` modules in-process with the model stubbed. No server, no API key, no
tokens spent.

First, a tiny log handler so the structured `event` tag on each log record is
visible - that tag is the whole point of these failure modes.

**Schema reject with retry-correct.** The stub returns `priority: "urgent"` on
the first call - not in `Literal["low","medium","high","critical"]`. Pydantic
rejects it, `validate_and_correct` logs `tool_arg_validation_failed`, appends
the validator's own error object as a corrective message, and re-calls. The
second call comes back valid.


In [ ]:
# In-process failure demos -- no server, no API key, no tokens.
import os, json, asyncio, logging, httpx
os.environ.setdefault('OPENAI_API_KEY', 'sk-notebook-stub')   # stubs only; never sent

from unittest.mock import patch
from app.llm import validate_and_correct
from app.main import intake_generator
from app.schemas import IntakeRequest

class TagHandler(logging.Handler):
    """Print the structured `event` tag that rides on each log record."""
    def emit(self, record):
        tag = getattr(record, 'event', None)
        if tag:
            print(f'  LOG  event={tag:<30} {record.getMessage()}')

root = logging.getLogger()
root.handlers = [TagHandler()]
root.setLevel(logging.INFO)

# --- OpenAI-shaped stubs (same shape the tests use) ---
class FakeFunction:
    def __init__(self, name, arguments): self.name, self.arguments = name, arguments
class FakeToolCall:
    def __init__(self, function): self.function = function
class FakeMessage:
    def __init__(self, content=None, tool_calls=None):
        self.content, self.tool_calls = content, tool_calls
class FakeChoice:
    def __init__(self, finish_reason, message):
        self.finish_reason, self.message = finish_reason, message
class FakeResponse:
    def __init__(self, choices): self.choices = choices

def ticket_args(priority):
    return {'intent': {'type': 'order', 'order_id': 'ORD-1042'},
            'priority': priority, 'customer_id': 8821, 'attachments': []}

def stub_client(sequence):
    """Return a patcher whose model emits ticket args from `sequence`, in order."""
    calls = {'n': 0}
    async def create(*args, **kwargs):
        i = min(calls['n'], len(sequence) - 1)
        calls['n'] += 1
        return FakeResponse([FakeChoice(
            finish_reason='tool_calls',
            message=FakeMessage(tool_calls=[FakeToolCall(
                FakeFunction('extract_ticket', json.dumps(ticket_args(sequence[i]))))]),
        )])
    return create, calls

# --- Run 1: invalid priority, then a valid one -> the correction loop RECOVERS ---
create, calls = stub_client(['urgent', 'high'])
with patch('app.llm._client') as mock:
    mock.return_value.chat.completions.create = create
    ticket, telemetry = await validate_and_correct('order ORD-1042 is missing')

print()
print('model calls      :', calls['n'])
print('ticket priority  :', ticket.priority, ' <- corrected from "urgent"')
print('telemetry        :', telemetry, ' <- correction_count 1')


A correction loop that fires and never converges is just a slower way to fail.
That is what the cap is for: `schema_correction_max_attempts = 2`. Below it, the
model gets another chance with the validator's error in hand. Above it, the
failure mode has changed - the schema is harder than the prompt explains, and
that needs a prompt edit, not another retry.

Here the stub is stubborn: it returns `"urgent"` every single time.


In [ ]:
# --- Run 2: the model NEVER corrects -> schema_correction_exhausted, ticket is None ---
create, calls = stub_client(['urgent'])
with patch('app.llm._client') as mock:
    mock.return_value.chat.completions.create = create
    ticket, telemetry = await validate_and_correct('order ORD-1042 is missing')

print()
print('model calls :', calls['n'], ' <- capped: 1 original + 2 corrections')
print('ticket      :', ticket, ' <- None: the gate held, nothing was routed')
print('telemetry   :', telemetry)


---
## 8 · Failure Mode - Mid-Stream Routing Failure (`tool_impl_transient_failed`)

`route_to_team` is the side-effect tool, and it is the one place `with_tool_retry`
is **actually applied** (`_route` in `main.py`). Here the routing service times
out on the first call.

The field frames (`intent`, `priority`, `customer_id`) have already gone out.
Tenacity absorbs the timeout, backs off, retries, and the client sees `routed`
arrive a beat later - no error, no interruption.

Note the asymmetry with OrderBot: TicketStream's `retry.py` has no `before_sleep`
hook, so a retry that *succeeds* is silent. The `tool_impl_transient_failed` tag
is emitted by `intake_generator` only when the retries are **exhausted** - see the
next cell. A silent successful retry is exactly the blind spot that hook exists
to close.


In [ ]:
# --- The routing service is flaky: it times out once, then succeeds ---
from app.schemas import TicketSchema, OrderIntent
from app.tools import route_to_team as real_route

TICKET = TicketSchema(intent=OrderIntent(order_id='ORD-1042'),
                      priority='high', customer_id=8821)

async def fake_validate(msg, model_name=None):
    return TICKET, {'correction_count': 0, 'attempts': 1}

hits = {'n': 0}
def flaky_route(team, priority):
    hits['n'] += 1
    if hits['n'] == 1:
        raise httpx.TimeoutException('routing-service timed out')
    return real_route(team, priority)

async def drive():
    frames = []
    async for f in intake_generator(IntakeRequest(message='Where is ORD-1042?')):
        frames.append(f)
    return [f.split('event: ')[1].split('\n')[0] for f in frames]

with patch('app.main.validate_and_correct', fake_validate), \
     patch('app.main.route_to_team', flaky_route):
    events = await drive()

print()
print('routing attempts :', hits['n'], ' <- tenacity retried after a bounded backoff')
print('SSE events       :', events)
print('                    the client saw "routed" -- just later. No failure frame.')


If the routing service is not flaky but **dead**, every attempt fails, the
exception propagates out of `_route`, and `intake_generator` catches it and
yields `event: validation_failed` with `stage: "post_validate"` - which is how
you tell a routing failure apart from a schema failure on the wire.


In [ ]:
# --- The routing service is dead: every attempt fails ---
def dead_route(team, priority):
    raise httpx.TimeoutException('routing-service is down')

with patch('app.main.validate_and_correct', fake_validate), \
     patch('app.main.route_to_team', dead_route):
    events = await drive()

print()
print('SSE events :', events)
print('             no "routed" frame -- validation_failed carries stage=post_validate.')
print()
print('Alert on a rising tool_impl_transient_failed rate. Retries hide a degrading')
print('dependency right up until they stop working.')


---
## 9 · Failure Mode - Model Emits Prose (`unexpected_text_response`)

The model answers in text instead of calling `extract_ticket`. `_aextract_once`
returns `(None, None, "stop")`, and `validate_and_correct` gives up
**immediately** - it does not retry. Re-prompting a model that just ignored the
tool list with the same messages will not help; the prompt needs fixing, not
another call.

`intake_generator` then yields `validation_failed` and `done`. **No routing
happens.** That is the safety gate: the side-effect tool never fires without a
valid `TicketSchema`.


In [ ]:
# --- The model responds in prose instead of calling the tool ---
async def just_talks(*args, **kwargs):
    return FakeResponse([FakeChoice(
        finish_reason='stop',
        message=FakeMessage(content="I'd be happy to help with your order!"),
    )])

routed_calls = {'n': 0}
def counting_route(team, priority):
    routed_calls['n'] += 1
    return real_route(team, priority)

# Note: validate_and_correct is NOT stubbed here -- we stub the model underneath it,
# so the real extraction path runs and the real gate decides.
with patch('app.llm._client') as mock, patch('app.main.route_to_team', counting_route):
    mock.return_value.chat.completions.create = just_talks
    frames = []
    async for f in intake_generator(IntakeRequest(message='Where is ORD-1042?')):
        frames.append(f)
    events = [f.split('event: ')[1].split('\n')[0] for f in frames]

print()
print('SSE events          :', events)
print('route_to_team calls :', routed_calls['n'], ' <- zero. The gate held.')
print()
print('A spike in unexpected_text_response correlates with a provider model update,')
print('not a tool issue. Retest SYSTEM_PROMPT against the new model version.')


---
## 10 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs -- try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [12]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))